# Persian Poetry Semantic Similarity Benchmark

This notebook evaluates LLMs on their ability to identify semantic outliers in Persian poetry couplets.

## Experiment Overview
- **Dataset**: Gherabat book questions (outlier detection tasks)
- **Task**: Given 4 poetry couplets, identify which one has a different conceptual meaning
- **Models**: Multiple LLMs via OpenRouter API
- **Approach**: Zero-shot or Few-shot prompting
- **Technology**: Structured Outputs (JSON Schema) for reliable, type-safe responses

## 1. Setup and Imports

In [1]:
import json
import os
import logging
import time
import asyncio
import random
from threading import Lock
from openai import OpenAI, AsyncOpenAI, RateLimitError, APIError
from dotenv import load_dotenv
import csv
import re
import pandas as pd
from pathlib import Path
from tqdm.asyncio import tqdm as atqdm
from tqdm import tqdm

# Load environment variables
load_dotenv(dotenv_path="../.env")

print("✅ Dependencies loaded successfully")

✅ Dependencies loaded successfully


## 2. Configuration

In [ ]:
# --- Configuration ---
QUESTIONS_FILE_PATH = "../data/gherabat-book/questions-outliers.json"
ANSWER_KEYS_FILE_PATH = "../data/gherabat-book/answer_keys.json"
LOG_FILE_PATH = "experiment-results.log"
CSV_RESULT_FILE_PATH = "experiment-answers.csv"
API_LOGS_FILE_PATH = "api-requests-responses.jsonl"  # JSONL file for all API requests/responses
REASONING_LOGS_DIR = "reasoning-outputs"  # Directory for reasoning tokens from thinking models
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

# Experiment mode: 'zero-shot' or 'few-shot'
EXPERIMENT_MODE = 'few-shot'  # Change to 'few-shot' for few-shot experiment

# Test mode: if True, will only test with 3 random questions (useful for testing/debugging)
TEST_MODE = False  # Set to True to run quick tests with only 3 random questions

# Resume mode: if True, will continue from existing results; if False, will start fresh
RESUME_MODE = False  # Set to False to start from scratch (will overwrite existing results)

# Concurrency settings
MAX_CONCURRENT_REQUESTS = 5  # Maximum number of concurrent API requests (up to 5 per OpenRouter limits)

# Per-model question limits (useful for expensive models)
# Format: {"model_name": max_questions}
# Models not listed here will use DEFAULT_QUESTION_LIMIT
MODEL_QUESTION_LIMITS = {
    "google/gemini-2.5-pro": 100,
    "deepseek/deepseek-r1-0528": 100,
    "anthropic/claude-haiku-4.5": 100,
    "moonshotai/kimi-k2-thinking": 100,
    "qwen/qwen3-next-80b-a3b-thinking": 100,
    "openai/gpt-oss-120b": 100,
    "openai/gpt-5-mini": 100,
    "openai/gpt-4.1-mini": 100,
}

# Default question limit for models not in MODEL_QUESTION_LIMITS
# Set to None for no limit (process all questions)
DEFAULT_QUESTION_LIMIT = 200  # None = no limit

# List of models to test
MODELS_TO_TEST = [
    "google/gemini-2.5-flash",
    "google/gemini-2.5-pro",
    "google/gemma-3-12b-it",
    "google/gemma-3-27b-it",
    "google/gemma-3-4b-it",
    "google/gemma-3n-e4b-it",
    "x-ai/grok-4-fast",
    "x-ai/grok-3-mini",
    "deepseek/deepseek-chat-v3.1",
    "deepseek/deepseek-r1-0528",
    "z-ai/glm-4.6",
    "openai/gpt-5-mini",
    "openai/gpt-4.1-mini",
    "openai/gpt-oss-20b",
    "openai/gpt-oss-120b",
    "meta-llama/llama-4-maverick",
    "meta-llama/llama-3.3-70b-instruct",
    "meta-llama/llama-3.1-8b-instruct",
    "moonshotai/kimi-k2-0905",
    "moonshotai/kimi-k2-thinking",
    "qwen/qwen3-32b",
    "qwen/qwen3-14b",
    "qwen/qwen3-235b-a22b",
    "qwen/qwen3-next-80b-a3b-instruct",
    "qwen/qwen3-next-80b-a3b-thinking",
    "qwen/qwen-2.5-72b-instruct",
    "qwen/qwen-2.5-7b-instruct",
    "mistralai/mistral-medium-3.1",
    "mistralai/ministral-8b",
    "anthropic/claude-haiku-4.5",
    "cohere/command-r-08-2024",
]

# Create reasoning logs directory if it doesn't exist
Path(REASONING_LOGS_DIR).mkdir(parents=True, exist_ok=True)

print(f"📝 Experiment mode: {EXPERIMENT_MODE}")
print(f"🧪 Test mode: {'ENABLED (3 random questions only)' if TEST_MODE else 'DISABLED (all questions)'}")
print(f"🔄 Resume mode: {'ENABLED' if RESUME_MODE else 'DISABLED (will start fresh)'}")
print(f"⚡ Concurrent requests: {MAX_CONCURRENT_REQUESTS}")
print(f"🤖 Testing {len(MODELS_TO_TEST)} models")
print(f"🎯 Question limits: {len(MODEL_QUESTION_LIMITS)} models with custom limits, default={DEFAULT_QUESTION_LIMIT}")
print(f"📊 Questions file: {QUESTIONS_FILE_PATH}")
print(f"🔑 Answer keys file: {ANSWER_KEYS_FILE_PATH}")
print(f"📝 API logs: {API_LOGS_FILE_PATH}")
print(f"🧠 Reasoning logs directory: {REASONING_LOGS_DIR}")

📝 Experiment mode: few-shot
🧪 Test mode: DISABLED (all questions)
🔄 Resume mode: DISABLED (will start fresh)
⚡ Concurrent requests: 5
🤖 Testing 31 models
🎯 Question limits: 8 models with custom limits, default=200
📊 Questions file: ../dataset/gherabat-book/questions-outliers.json
🔑 Answer keys file: ../dataset/gherabat-book/answer_keys.json
📝 API logs: api-requests-responses.jsonl
🧠 Reasoning logs directory: reasoning-outputs


## 3. System Prompts

In [3]:
# Zero-shot prompt (updated for simple digit response)
ZERO_SHOT_PROMPT = "You are an AI assistant analyzing Persian poetry couplets. Identify the outlier option based on concept/message."

# Few-shot prompt with examples (updated for simple digit response)
FEW_SHOT_PROMPT = """You are an expert literary critic with a deep understanding of Persian poetry, its cultural nuances, and its stylistic features. Your task is to analyze a set of poetic options—each option presenting two parts of a couplet—and identify the one option that deviates in conceptual meaning or thematic message from the others. Focus exclusively on the underlying concepts, disregarding stylistic or linguistic differences.

For example:

---
Options:
1. طریق عشق پرآشوب و فتنه است ای دل - بیفتد آن که در این راه با شتاب رود
2. گر نور عشق حق به دل و جانت اوفتد - بالله از آفتاب فلک خوبتر شوی
3. شکوه عشق نگه کن که موی مجنون را - فلک به شعشعه آفتاب، شانه کند
4. فرزانه درآید به پری خانه مقصود - هر کس که در این بادیه دیوانه عشق است

Correct answer: 1

(Option 1 warns against hastily pursuing the turbulent path of love, whereas the other options present love as an uplifting force)

---
Options:
1. شمشیر نیک از آهن بد چون کند کسی؟ - ناکس تربیت نشود ای حکیم کس
2. سگ به دریای هفت گانه بشوی - که چو تر شد پلیدتر باشد
3. ز وحشی نیاید که مردم شود - به سعی اندر او تربیت گم شود
4. سگ اصحاب کهف روزی چند - پی نیکان گرفت و مردم شد

Correct answer: 4

(Option 4 emphasizes the significant impact of upbringing, unlike the other options which imply that upbringing makes little difference)

---
Options:
1. هر چند خوشگوار بود باده غرور - زین می فزون از سنگ نگه دار شیشه را
2. از ساده دلی هر که دهد پند به مغرور - بیدار به افسانه کند خواب گران را
3. کبر مفروش به مردم که به میزان نظر - زود گردد سبک آن کس که بود سنگین تر
4. خاک بر فرقش اگر از کبر سر بالا کند - هر که داند بازگشت او به غیر از خاک نیست

Correct answer: 2

(The meaning of option 2 is the ineffectiveness of giving advice to the arrogant, while the common meaning of the other options is the recommendation to avoid arrogance)"""

# Select prompt based on experiment mode
SYSTEM_PROMPT = FEW_SHOT_PROMPT if EXPERIMENT_MODE == 'few-shot' else ZERO_SHOT_PROMPT

print(f"✅ Using {EXPERIMENT_MODE} prompt (simple digit response)")

✅ Using few-shot prompt (simple digit response)


## 4. Logging Setup

In [4]:
logging.basicConfig(
    level=logging.WARNING,  # Changed from INFO to WARNING - only show warnings and errors
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE_PATH),
        logging.StreamHandler()
    ]
)

print("✅ Logging configured (errors/warnings only)")

✅ Logging configured (errors/warnings only)


## 5. Helper Functions

In [5]:
import datetime

# Lock for thread-safe file writing
api_log_lock = Lock()
reasoning_log_lock = Lock()


def log_api_request_response(model_id, question_id, request_data, response_data, reasoning_data=None):
    """
    Logs API request and response to a JSONL file.

    Args:
        model_id: The model identifier
        question_id: The question ID
        request_data: The request payload sent to API
        response_data: The raw response from API
        reasoning_data: Optional reasoning/thinking data extracted from response
    """
    log_entry = {
        "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
        "model_id": model_id,
        "question_id": question_id,
        "request": request_data,
        "response": response_data,
        "reasoning": reasoning_data
    }

    with api_log_lock:
        try:
            with open(API_LOGS_FILE_PATH, 'a', encoding='utf-8') as f:
                f.write(json.dumps(log_entry, ensure_ascii=False) + '\n')
        except Exception as e:
            # Only print errors to console
            print(f"⚠️  Failed to log API request/response: {e}")


def save_reasoning_tokens(model_id, question_id, reasoning_text, reasoning_details=None):
    """
    Saves reasoning/thinking tokens to a separate file.

    Args:
        model_id: The model identifier
        question_id: The question ID
        reasoning_text: The reasoning text from message.reasoning
        reasoning_details: The reasoning_details array from the response
    """
    if not reasoning_text and not reasoning_details:
        return

    # Create filename with model and question IDs
    safe_model_id = model_id.replace('/', '_').replace(':', '_')
    filename = f"{safe_model_id}_q{question_id}.json"
    filepath = Path(REASONING_LOGS_DIR) / filename

    reasoning_data = {
        "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
        "model_id": model_id,
        "question_id": question_id,
        "reasoning_text": reasoning_text,
        "reasoning_details": reasoning_details
    }

    with reasoning_log_lock:
        try:
            with open(filepath, 'w', encoding='utf-8') as f:
                json.dump(reasoning_data, f, ensure_ascii=False, indent=2)
            # Don't print success messages
        except Exception as e:
            # Only print errors to console
            print(f"⚠️  Failed to save reasoning tokens: {e}")


def extract_answer_digit(text: str) -> int:
    """
    Extract answer digit (1, 2, 3, or 4) from text response.
    Handles cases where models include reasoning/thinking before the answer.

    Args:
        text: Text that may contain the answer digit

    Returns:
        Integer answer (1-4) or None if not found
    """
    if not text:
        return None

    # Look for standalone digit 1, 2, 3, or 4
    # Try to find it at the end first (most common case)
    matches = re.findall(r'[1-4]', text)

    if matches:
        # Return the last occurrence (usually the final answer)
        answer = int(matches[-1])
        if answer in [1, 2, 3, 4]:
            return answer

    return None


def load_questions(file_path):
    """Loads questions from the Gherabat dataset JSON file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # Extract all questions from all pages
        questions_list = []
        if 'pages' in data:
            for page in data['pages']:
                if 'data' in page and 'questions' in page['data']:
                    questions_list.extend(page['data']['questions'])

        if not questions_list:
            logging.error(f"No questions found in {file_path}")
            return None

        logging.info(f"Successfully loaded {len(questions_list)} questions from {file_path}")
        return questions_list  # Return all questions (TEST_MODE will handle sampling if needed)
    except FileNotFoundError:
        logging.error(f"Error: JSON file not found at {file_path}")
        return None
    except json.JSONDecodeError as e:
        logging.error(f"Error: Could not decode JSON from {file_path}. Details: {e}")
        return None
    except Exception as e:
        logging.error(f"An unexpected error occurred loading JSON: {e}")
        return None


def load_answer_keys(file_path):
    """Loads answer keys from JSON file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            answer_keys = json.load(f)
        logging.info(f"Successfully loaded {len(answer_keys)} answer keys from {file_path}")
        return answer_keys
    except FileNotFoundError:
        logging.error(f"Error: Answer keys file not found at {file_path}")
        return None
    except json.JSONDecodeError as e:
        logging.error(f"Error: Could not decode JSON from {file_path}. Details: {e}")
        return None
    except Exception as e:
        logging.error(f"An unexpected error occurred loading answer keys: {e}")
        return None


def load_existing_results(file_path):
    """
    Loads existing results from CSV file for resume functionality.
    Returns a set of (model_name, question_id) tuples that have been completed.
    """
    completed = set()

    if not os.path.exists(file_path):
        logging.info(f"No existing results file found at {file_path}. Starting fresh.")
        return completed

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                model_name = row.get('model_name')
                question_id = row.get('question_id')

                if model_name and question_id:
                    completed.add((model_name, str(question_id)))

        logging.info(f"Loaded {len(completed)} existing results from {file_path}")
        return completed

    except Exception as e:
        logging.warning(f"Error loading existing results from {file_path}: {e}")
        logging.warning("Starting fresh.")
        return set()


def format_prompt(question_data):
    """Formats the question into a prompt for the AI."""
    if 'options' not in question_data or not isinstance(question_data['options'], list):
        logging.error(f"Invalid question structure or missing 'options' for question id: {question_data.get('id', 'N/A')}")
        return None

    prompt = "Analyze the conceptual meaning of the following options:\n\n"
    prompt += "Options:\n"

    try:
        sorted_options = sorted(question_data['options'], key=lambda x: int(x.get('label', 0)))

        if not sorted_options:
            logging.error(f"No valid options found for question {question_data.get('id', 'N/A')}")
            return None

        for option in sorted_options:
            if not all(k in option for k in ['label', 'mesra1', 'mesra2']):
                logging.warning(f"Skipping malformed option in question {question_data.get('id', 'N/A')}")
                continue
            prompt += f"{option['label']}. {option['mesra2']} - {option['mesra1']}\n"

    except (TypeError, ValueError, KeyError) as e:
        logging.error(f"Error formatting options for question {question_data.get('id', 'N/A')}: {e}")
        return None

    prompt += "\nIdentify the single option that has a different concept and message from the others."
    prompt += "\n\nRespond with ONLY the number (1, 2, 3, or 4) of your answer. No explanation needed."

    return prompt


async def get_model_response_async(client, model_id, prompt, semaphore, question_id):
    """
    Async version: Sends the prompt to the specified model.
    Uses semaphore to limit concurrent requests.
    Returns a tuple: (answer: int | None, completion_tokens: int | None, error_msg: str | None)

    This function:
    - Logs all requests to file
    - Only prints errors to console
    - Enables reasoning tokens for thinking models
    - Saves reasoning tokens to separate files
    """
    if not prompt:
        return None, None, "ERROR: Invalid Prompt"

    async with semaphore:  # Limit concurrent requests
        try:
            # Prepare request payload (no response_format - simple text response)
            request_payload = {
                "model": model_id,
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt}
                ],
                "stream": False,
                "temperature": 0,
            }

            # Enable reasoning for all models (will be ignored by non-reasoning models)
            extra_body = {
                "reasoning": {}  # Enable reasoning with default settings
            }

            completion = await client.chat.completions.create(
                **request_payload,
                extra_body=extra_body
            )

            tokens_used = None
            answer = None
            error_msg = None
            reasoning_text = None
            reasoning_details = None

            if hasattr(completion, 'usage') and hasattr(completion.usage, 'completion_tokens'):
                tokens_used = completion.usage.completion_tokens

            if completion.choices and len(completion.choices) > 0:
                message = completion.choices[0].message
                content = message.content
                finish_reason = completion.choices[0].finish_reason

                # Extract reasoning if available
                reasoning_text = getattr(message, 'reasoning', None)
                reasoning_details = getattr(message, 'reasoning_details', None)

                # Save reasoning tokens silently (errors will be printed)
                if reasoning_text or reasoning_details:
                    logging.info(f"Model {model_id} produced reasoning tokens for question {question_id}")
                    save_reasoning_tokens(model_id, question_id, reasoning_text, reasoning_details)

                if content is None:
                    # Print error to console
                    print(f"❌ Model {model_id} Q{question_id}: No content (finish_reason={finish_reason})")
                    logging.warning(f"Received None content from {model_id}. Finish Reason: {finish_reason}")
                    error_msg = "ERROR: None Content Received"
                else:
                    # Extract the answer digit from the content
                    answer = extract_answer_digit(content)

                    if answer is None:
                        # Print error to console
                        print(f"❌ Model {model_id} Q{question_id}: Could not extract answer from: '{content[:100]}'")
                        logging.error(f"Failed to extract answer digit from {model_id}. Content (first 200 chars): '{content[:200]}'")
                        error_msg = f"ERROR: Could not extract answer digit from response"
                    elif answer not in [1, 2, 3, 4]:
                        # Print error to console
                        print(f"❌ Model {model_id} Q{question_id}: Invalid answer {answer}")
                        logging.warning(f"Model {model_id} returned invalid answer: {answer}")
                        error_msg = f"ERROR: Invalid answer value: {answer}"
                        answer = None

                    if finish_reason == 'length':
                        print(f"⚠️  Model {model_id} Q{question_id}: Response truncated")
                        logging.warning(f"Model {model_id} response may be truncated (finish_reason='length').")
            else:
                print(f"❌ Model {model_id} Q{question_id}: Malformed response")
                logging.warning(f"Received unexpected response object structure from {model_id}.")
                error_msg = "ERROR: Malformed Response Object"

            # Log the API request and response (to file only)
            response_dict = {
                "choices": [
                    {
                        "message": {
                            "role": message.role if completion.choices else None,
                            "content": message.content if completion.choices else None,
                            "reasoning": reasoning_text,
                            "reasoning_details": reasoning_details
                        },
                        "finish_reason": finish_reason if completion.choices else None
                    }
                ] if completion.choices else [],
                "usage": {
                    "completion_tokens": tokens_used,
                    "prompt_tokens": getattr(completion.usage, 'prompt_tokens', None) if hasattr(completion, 'usage') else None,
                    "total_tokens": getattr(completion.usage, 'total_tokens', None) if hasattr(completion, 'usage') else None
                } if hasattr(completion, 'usage') else None
            }

            log_api_request_response(
                model_id,
                question_id,
                request_payload,
                response_dict,
                {"reasoning_text": reasoning_text, "reasoning_details": reasoning_details} if (reasoning_text or reasoning_details) else None
            )

            return answer, tokens_used, error_msg

        except RateLimitError as e:
            print(f"⚠️  Model {model_id} Q{question_id}: Rate limit hit, waiting 60s...")
            logging.warning(f"Rate limit hit for model {model_id}: {e}")
            await asyncio.sleep(60)

            log_api_request_response(
                model_id,
                question_id,
                {"error": "Rate limit hit"},
                {"error": str(e)}
            )

            return None, None, f"ERROR: Rate Limit Hit - {e}"

        except APIError as e:
            err_msg = e.message or str(e.body)
            print(f"❌ Model {model_id} Q{question_id}: API Error ({e.status_code}) - {err_msg[:100]}")
            logging.error(f"API Error for model {model_id} (Code: {e.status_code}): {err_msg}")

            log_api_request_response(
                model_id,
                question_id,
                {"error": "API Error"},
                {"error": err_msg, "status_code": e.status_code}
            )

            if e.status_code == 429:
                return None, None, f"ERROR: API Rate Limit (429) - {err_msg}"
            elif "context_length_exceeded" in err_msg:
                return None, None, f"ERROR: Context Length Exceeded - {err_msg}"
            elif e.status_code == 500 and "Model is overloaded" in err_msg:
                return None, None, f"ERROR: Model Overloaded (500) - {err_msg}"
            elif e.status_code == 400:
                if "does not exist" in err_msg:
                    return None, None, f"ERROR: Model Not Found (400) - {err_msg}"
                else:
                    return None, None, f"ERROR: API Error (400) - {err_msg}"
            else:
                return None, None, f"ERROR: API Error ({e.status_code}) - {err_msg}"

        except Exception as e:
            print(f"❌ Model {model_id} Q{question_id}: Unexpected error - {type(e).__name__}: {str(e)[:100]}")
            logging.error(f"Unexpected error calling model {model_id}: {type(e).__name__} - {e}")

            log_api_request_response(
                model_id,
                question_id,
                {"error": "Unexpected error"},
                {"error": str(e), "type": type(e).__name__}
            )

            return None, None, f"ERROR: Unexpected - {type(e).__name__} - {e}"


print("✅ Helper functions defined (with async support, reasoning extraction, and API logging)")

✅ Helper functions defined (with async support, reasoning extraction, and API logging)


## 6. Load Data

In [6]:
# Load questions and answer keys
questions = load_questions(QUESTIONS_FILE_PATH)
answer_keys = load_answer_keys(ANSWER_KEYS_FILE_PATH)

if not questions:
    raise ValueError("No questions loaded or error loading questions.")

if not answer_keys:
    raise ValueError("No answer keys loaded or error loading answer keys.")

# Apply test mode: randomly select 5 questions if TEST_MODE is enabled
if TEST_MODE:
    original_count = len(questions)
    if original_count > 3:
        questions = random.sample(questions, 3)
        selected_ids = [q.get('id') for q in questions]
        logging.info(f"TEST MODE: Randomly selected 3 questions from {original_count} total questions")
        logging.info(f"TEST MODE: Selected question IDs: {selected_ids}")
        print(f"\n🧪 TEST MODE ACTIVE: Randomly selected 3 questions (IDs: {selected_ids})")
    else:
        logging.warning(f"TEST MODE: Only {original_count} questions available, using all of them")
        print(f"\n🧪 TEST MODE ACTIVE: Using all {original_count} available questions")

print(f"\n📊 Dataset Statistics:")
print(f"Total questions: {len(questions)}")
print(f"Total answer keys: {len(answer_keys)}")
print(f"\nSample question ID: {questions[0].get('id')}")
print(f"Sample question stem: {questions[0].get('stem')}")


📊 Dataset Statistics:
Total questions: 591
Total answer keys: 1500

Sample question ID: 6
Sample question stem: کدام گزینه مفهومی متفاوت با سایر گزینه ها دارد؟


## 7. Initialize OpenAI Client

In [7]:
if not OPENROUTER_API_KEY:
    raise ValueError("FATAL: OPENROUTER_API_KEY environment variable not set.")

try:
    # Initialize AsyncOpenAI client for concurrent requests
    async_client = AsyncOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
        timeout=60.0,  # 1 minute timeout per request (longer for concurrent operations)
    )
    logging.info("AsyncOpenAI client initialized with 60 second timeout.")
    print("✅ AsyncOpenAI client initialized for concurrent execution")
except Exception as e:
    raise RuntimeError(f"FATAL: Failed to initialize AsyncOpenAI client: {e}")

✅ AsyncOpenAI client initialized for concurrent execution


## 8. Run Experiment

In [8]:
# Async function to process a single question
async def process_question(client, model_id, question, answer_keys, semaphore, csv_lock, csv_writer, csvfile):
    """Process a single question asynchronously"""
    q_id = question.get("id")

    # Format the prompt
    prompt = format_prompt(question)

    if not prompt:
        error_msg = "ERROR: Prompt Formatting Failed"
        model_answer = None
        tokens_used = None
    else:
        # Get response from the model
        model_answer, tokens_used, error_msg = await get_model_response_async(
            client, model_id, prompt, semaphore, q_id
        )

    # Get correct answer from answer keys
    correct_answer = answer_keys.get(str(q_id), None)

    # Check if answer is correct
    is_correct = False
    if model_answer is not None and correct_answer is not None:
        is_correct = int(model_answer) == int(correct_answer)

    # Prepare values for CSV
    csv_answer = model_answer if model_answer is not None else ""
    csv_tokens = tokens_used if tokens_used is not None else ""

    # Write result to CSV (thread-safe)
    async with csv_lock:
        csv_writer.writerow([model_id, q_id, csv_answer, correct_answer, is_correct, csv_tokens])
        csvfile.flush()  # Ensure data is written immediately

    # Don't log successful requests to console
    if error_msg:
        logging.error(f"Model: {model_id} | Question: {q_id} | Status: Failed | Detail: {error_msg}")

    return {
        'question_id': q_id,
        'is_correct': is_correct,
        'error': error_msg
    }


# Main async function to run the experiment
async def run_experiment_async():
    """Main async function to run the experiment with concurrent requests and progress bar"""

    # Load existing results if resume mode is enabled
    completed_pairs = set()
    if RESUME_MODE:
        completed_pairs = load_existing_results(CSV_RESULT_FILE_PATH)
        if completed_pairs:
            print(f"\n🔄 Resume mode: Found {len(completed_pairs)} existing results")
        else:
            print(f"\n🔄 Resume mode enabled but no existing results found. Starting fresh.")
    else:
        print(f"\n🆕 Starting fresh (RESUME_MODE = False). Will overwrite existing results.")

    # Calculate total work
    total_combinations = len(MODELS_TO_TEST) * len(questions)
    already_completed = len(completed_pairs)
    remaining_work = total_combinations - already_completed

    print(f"\n📊 Work Summary:")
    print(f"   Total combinations: {total_combinations} ({len(MODELS_TO_TEST)} models × {len(questions)} questions)")
    print(f"   Already completed: {already_completed}")
    print(f"   Remaining: {remaining_work}\n")

    # Create semaphore for limiting concurrent requests
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

    # Create lock for thread-safe CSV writing
    csv_lock = asyncio.Lock()

    # Determine file mode
    file_mode = 'a' if (RESUME_MODE and completed_pairs) else 'w'
    write_header = (file_mode == 'w')

    try:
        with open(CSV_RESULT_FILE_PATH, file_mode, newline='', encoding='utf-8') as csvfile:
            csv_writer = csv.writer(csvfile)

            # Write header only if starting fresh
            if write_header:
                csv_writer.writerow(['model_name', 'question_id', 'model_answer', 'correct_answer', 'is_correct', 'completion_tokens'])
                logging.info(f"Created new results file: {CSV_RESULT_FILE_PATH}")
            else:
                logging.info(f"Appending to existing results file: {CSV_RESULT_FILE_PATH}")

            total_questions = len(questions)

            # Iterate through each model
            for model_index, model_id in enumerate(MODELS_TO_TEST):
                # Get question limit for this model
                question_limit = MODEL_QUESTION_LIMITS.get(model_id, DEFAULT_QUESTION_LIMIT)

                print(f"\n🤖 Model {model_index+1}/{len(MODELS_TO_TEST)}: {model_id}")
                if question_limit is not None:
                    print(f"   🎯 Question limit: {question_limit}")

                # Filter questions that need to be processed
                questions_to_process = [
                    q for q in questions
                    if not (RESUME_MODE and (model_id, str(q.get('id'))) in completed_pairs)
                ]

                # Apply per-model question limit
                if question_limit is not None and len(questions_to_process) > question_limit:
                    original_count = len(questions_to_process)
                    questions_to_process = questions_to_process[:question_limit]
                    print(f"   📉 Limited from {original_count} to {question_limit} questions")

                skipped_count = total_questions - len(questions_to_process)
                if skipped_count > 0:
                    print(f"   ⏭️  Skipping {skipped_count} (completed or beyond limit)")

                if not questions_to_process:
                    print(f"   ✅ All questions already completed or limit reached")
                    continue

                start_time = time.time()

                # Create tasks for all questions
                tasks = [
                    process_question(
                        async_client, model_id, question, answer_keys,
                        semaphore, csv_lock, csv_writer, csvfile
                    )
                    for question in questions_to_process
                ]

                # Run tasks with progress bar
                results = []
                with tqdm(total=len(tasks), desc=f"   Progress", unit="q", ncols=80, leave=False) as pbar:
                    for coro in asyncio.as_completed(tasks):
                        result = await coro
                        results.append(result)
                        pbar.update(1)

                elapsed_time = time.time() - start_time

                # Calculate statistics
                success_count = sum(1 for r in results if isinstance(r, dict) and r.get('is_correct'))
                error_count = sum(1 for r in results if isinstance(r, dict) and r.get('error'))
                exception_count = sum(1 for r in results if isinstance(r, Exception))

                processed_count = len(questions_to_process)
                accuracy = (success_count / processed_count * 100) if processed_count > 0 else 0

                print(f"   ✓ Processed: {processed_count} | Correct: {success_count} | Accuracy: {accuracy:.1f}% | Time: {elapsed_time:.1f}s")
                if error_count > 0:
                    print(f"   ⚠️  Errors: {error_count}")

        print(f"\n{'='*60}")
        print(f"✅ EXPERIMENT COMPLETE!")
        print(f"{'='*60}")
        print(f"Results saved to: {CSV_RESULT_FILE_PATH}")

    except Exception as e:
        logging.error(f"An unexpected error occurred during main execution: {e}")
        import traceback
        logging.error(traceback.format_exc())
        raise


# Run the async experiment
try:
    await run_experiment_async()
except KeyboardInterrupt:
    print(f"\n\n⚠️  INTERRUPTED BY USER")
    print(f"Progress has been saved to: {CSV_RESULT_FILE_PATH}")
    logging.warning("Experiment interrupted by user. Progress saved.")


🆕 Starting fresh (RESUME_MODE = False). Will overwrite existing results.

📊 Work Summary:
   Total combinations: 18321 (31 models × 591 questions)
   Already completed: 0
   Remaining: 18321


🤖 Model 1/31: google/gemini-2.5-flash
   🎯 Question limit: 200
   📉 Limited from 591 to 200 questions
   ⏭️  Skipping 391 (completed or beyond limit)


   ✓ Processed: 200 | Correct: 104 | Accuracy: 52.0% | Time: 16.3s

🤖 Model 2/31: google/gemini-2.5-pro
   🎯 Question limit: 100
   📉 Limited from 591 to 100 questions
   ⏭️  Skipping 491 (completed or beyond limit)


   ✓ Processed: 100 | Correct: 67 | Accuracy: 67.0% | Time: 312.0s

🤖 Model 3/31: google/gemma-3-12b-it
   🎯 Question limit: 200
   📉 Limited from 591 to 200 questions
   ⏭️  Skipping 391 (completed or beyond limit)


   ✓ Processed: 200 | Correct: 62 | Accuracy: 31.0% | Time: 34.4s

🤖 Model 4/31: google/gemma-3-27b-it
   🎯 Question limit: 200
   📉 Limited from 591 to 200 questions
   ⏭️  Skipping 391 (completed or beyond limit)


   ✓ Processed: 200 | Correct: 76 | Accuracy: 38.0% | Time: 33.1s

🤖 Model 5/31: google/gemma-3-4b-it
   🎯 Question limit: 200
   📉 Limited from 591 to 200 questions
   ⏭️  Skipping 391 (completed or beyond limit)


...

❌ Model mistralai/mistral-medium-3.1 Q134: Malformed response
   ✓ Processed: 200 | Correct: 81 | Accuracy: 40.5% | Time: 945.4s
   ⚠️  Errors: 4

🤖 Model 29/31: mistralai/ministral-8b
   🎯 Question limit: 200
   📉 Limited from 591 to 200 questions
   ⏭️  Skipping 391 (completed or beyond limit)


   Progress: 100%|████████████████████████████▊| 199/200 [00:24<00:00, 15.91q/s]2025-11-09 03:46:34,179 - WARNING - Received unexpected response object structure from mistralai/ministral-8b.
2025-11-09 03:46:34,181 - ERROR - Model: mistralai/ministral-8b | Question: 458 | Status: Failed | Detail: ERROR: Malformed Response Object


❌ Model mistralai/ministral-8b Q458: Malformed response
   ✓ Processed: 200 | Correct: 47 | Accuracy: 23.5% | Time: 910.9s
   ⚠️  Errors: 1

🤖 Model 30/31: anthropic/claude-haiku-4.5
   🎯 Question limit: 100
   📉 Limited from 591 to 100 questions
   ⏭️  Skipping 491 (completed or beyond limit)


   ✓ Processed: 100 | Correct: 45 | Accuracy: 45.0% | Time: 22.1s

🤖 Model 31/31: cohere/command-r-08-2024
   🎯 Question limit: 200
   📉 Limited from 591 to 200 questions
   ⏭️  Skipping 391 (completed or beyond limit)


   ✓ Processed: 200 | Correct: 53 | Accuracy: 26.5% | Time: 17.0s

✅ EXPERIMENT COMPLETE!
Results saved to: experiment-answers.csv


## 9. Results Analysis

In [11]:
# Load and analyze results
df = pd.read_csv(CSV_RESULT_FILE_PATH)

print("\n📊 EXPERIMENT RESULTS\n" + "="*50)

# Calculate accuracy per model
model_stats = df.groupby('model_name').agg({
    'is_correct': ['sum', 'count', 'mean'],
    'completion_tokens': 'sum'
}).round(4)

model_stats.columns = ['correct_answers', 'total_questions', 'accuracy', 'total_tokens']
model_stats['accuracy_pct'] = (model_stats['accuracy'] * 100).round(2)
model_stats = model_stats.sort_values('accuracy', ascending=False)

print("\nModel Performance:")
print(model_stats[['correct_answers', 'total_questions', 'accuracy_pct', 'total_tokens']])

# Summary statistics
print("\n" + "="*50)
print(f"Best performing model: {model_stats.index[0]}")
print(f"Best accuracy: {model_stats['accuracy_pct'].iloc[0]:.2f}%")
print(f"Average accuracy across all models: {model_stats['accuracy_pct'].mean():.2f}%")

# Display the dataframe
display(df.head(20))


📊 EXPERIMENT RESULTS

Model Performance:
                                   correct_answers  total_questions  \
model_name                                                            
google/gemini-2.5-pro                           67              100   
moonshotai/kimi-k2-thinking                     59              100   
openai/gpt-5-mini                               57              100   
deepseek/deepseek-r1-0528                       55              100   
x-ai/grok-4-fast                               106              200   
google/gemini-2.5-flash                        104              200   
z-ai/glm-4.6                                   100              200   
x-ai/grok-3-mini                                99              200   
openai/gpt-oss-120b                             49              100   
anthropic/claude-haiku-4.5                      45              100   
qwen/qwen3-next-80b-a3b-thinking                44              100   
qwen/qwen3-14b                     

,model_name,question_id,model_answer,correct_answer,is_correct,completion_tokens
0,google/gemini-2.5-flash,70,1.0,1,True,1.0
1,google/gemini-2.5-flash,282,3.0,2,False,1.0
2,google/gemini-2.5-flash,363,4.0,4,True,1.0
3,google/gemini-2.5-flash,207,4.0,3,False,1.0
4,google/gemini-2.5-flash,467,4.0,4,True,1.0
5,google/gemini-2.5-flash,325,1.0,1,True,1.0
6,google/gemini-2.5-flash,155,4.0,4,True,1.0
7,google/gemini-2.5-flash,248,2.0,3,False,1.0
8,google/gemini-2.5-flash,420,4.0,4,True,1.0
9,google/gemini-2.5-flash,79,1.0,1,True,1.0


## 10. Visualization (Optional)

In [ ]:
# Visualize the comparison
if common_models:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # Plot 1: Side-by-side comparison
    comparison_df_sorted = comparison_df.sort_values('improved_accuracy', ascending=True)
    x = range(len(comparison_df_sorted))
    width = 0.35

    ax1.barh([i - width/2 for i in x], comparison_df_sorted['original_accuracy'],
             width, label='Original', color='steelblue', alpha=0.8)
    ax1.barh([i + width/2 for i in x], comparison_df_sorted['improved_accuracy'],
             width, label='Improved', color='coral', alpha=0.8)

    ax1.set_yticks(x)
    ax1.set_yticklabels(comparison_df_sorted['model'])
    ax1.set_xlabel('Accuracy (%)')
    ax1.set_title('Original vs Improved Accuracy by Model')
    ax1.legend()
    ax1.grid(axis='x', alpha=0.3)
    ax1.set_xlim(0, 100)

    # Plot 2: Improvement delta
    comparison_df_sorted2 = comparison_df.sort_values('absolute_improvement', ascending=True)
    colors = ['green' if x > 0 else 'red' for x in comparison_df_sorted2['absolute_improvement']]

    ax2.barh(range(len(comparison_df_sorted2)), comparison_df_sorted2['absolute_improvement'],
             color=colors, alpha=0.7)
    ax2.set_yticks(range(len(comparison_df_sorted2)))
    ax2.set_yticklabels(comparison_df_sorted2['model'])
    ax2.set_xlabel('Accuracy Improvement (percentage points)')
    ax2.set_title('Absolute Improvement by Model')
    ax2.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
    ax2.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("\n✅ Comparison visualization complete")